# Joint Digital × Middle-Income Trap — data-improvement notebook

This notebook does three things:

1. **Diagnoses** why your fixed-effects results are fragile (runs on your existing panel).
2. **Re-estimates** the F×G complementarity with a Mundlak / correlated-random-effects
   decomposition that splits each coefficient into *within-* and *between-*country effects
   (runs on your existing panel).
3. **Rebuilds** the FinTech index `F` (4 Findex components + annual IMF FAS) and the AI
   index `G` (annual GARI vintages + AI patents) so both have real within-country time
   variation (needs a few public files you download — links inline).

Root cause it targets: country FE identify off *within*-country variation, and in the
current panel `F` and `G` barely move within country (within/between SD ≈ 0.28 and 0.09).

> **Run order in Colab:** Runtime ▸ Run all. Sections 0–3 run on your uploaded panel.
> Sections 4–5 need the public source files; instructions are in those cells.

## 0. Setup

In [ ]:
!pip -q install linearmodels pycountry_convert
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from linearmodels.panel import PanelOLS, RandomEffects
import pycountry_convert as pc
pd.set_option("display.float_format", lambda v: f"{v:0.4f}")
print("ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.1/255.1 kB 23.5 MB/s eta 0:00:00
ready


### Load your panel

Run the cell and pick `final_panel_imputed.csv` from your machine. (Or set `PANEL_CSV`
to a path / Google-Drive-mounted file and skip the upload.)

In [ ]:
PANEL_CSV = "final_panel_imputed.csv"
try:
    df = pd.read_csv(PANEL_CSV)
except FileNotFoundError:
    from google.colab import files
    up = files.upload()
    PANEL_CSV = list(up.keys())[0]
    df = pd.read_csv(PANEL_CSV)

CONTROLS = ["log_gdp","trade_pct_gdp","inflation","investment_pct_gdp",
            "pop_growth","credit_to_private_pct_gdp","secondary_enrollment"]
Cl = [f"{c}_l" for c in CONTROLS]
star = lambda p: "***" if p<.01 else "**" if p<.05 else "*" if p<.1 else ""
print(df.shape, "| countries:", df.iso3.nunique(), "| years:", int(df.year.min()), "-", int(df.year.max()))

Saving final_panel_imputed__5_.csv to final_panel_imputed__5_.csv
(3255, 139) | countries: 217 | years: 2010 - 2024


## 1. The diagnosis — how much do F and G actually move within country?

Country FE delete all between-country variation. If the within/between SD ratio is small,
the estimator is identifying off a thin slice of the data. This is the whole problem.

In [ ]:
def wb_ratio(v):
    w = df.groupby("iso3")[v].std().mean()
    b = df.groupby("iso3")[v].mean().std()
    return w, b, (w/b if b else np.nan)

rows = []
for v in ["F","G","findex_account_t_d","ai_pubs_log","Z_readiness","Z_adoption","Z_activity"]:
    if v in df.columns:
        w,b,r = wb_ratio(v)
        rows.append((v, round(w,4), round(b,4), round(r,4)))
print(pd.DataFrame(rows, columns=["var","within_SD","between_SD","within/between"]).to_string(index=False))
print("\nZ_readiness / Z_adoption with within-SD = 0 are single snapshots -> deleted by country FE.")

               var  within_SD  between_SD  within/between
                 F     0.2653      0.9451          0.2807
                 G     0.0801      0.8680          0.0923
findex_account_t_d     0.1021      0.2760          0.3701
       ai_pubs_log     0.5217      2.7260          0.1914
       Z_readiness     0.0000      1.0043          0.0000
        Z_adoption     0.0000      0.9279          0.0000
        Z_activity     0.1949      1.0182          0.1914

Z_readiness / Z_adoption with within-SD = 0 are single snapshots -> deleted by country FE.


## 2. Baseline (reproduce your headline) + Mundlak / CRE decomposition

`fe()` reproduces your two-way FE headline. `cre()` adds country-mean (Mundlak) terms and
estimates by random effects: the coefficient on `x_it` is the **within** effect, and
`x_it + mean` is the **between** effect; the p-value on the mean term tests whether they
differ (a Hausman-type test).

In [ ]:
def region(i):
    try: return pc.country_alpha2_to_continent_code(pc.country_alpha3_to_country_alpha2(i))
    except: return None
df["region"] = df["iso3"].map(region)

def fe(dep, regs):
    d = df.dropna(subset=[dep]+regs).copy().set_index(["iso3","year"])
    f = f"{dep} ~ 1 + " + " + ".join(regs) + " + EntityEffects + TimeEffects"
    return PanelOLS.from_formula(f, data=d, drop_absorbed=True).fit(cov_type="clustered", cluster_entity=True)

KEY = ["F_l","G_l","FG_l","MI_l","F_MI_l","G_MI_l","FG_MI_l"]
for v in KEY:
    df[v+"_bar"] = df.groupby("iso3")[v].transform("mean")

def cre(dep, regs):
    bars = [v+"_bar" for v in regs]
    d = df.dropna(subset=[dep]+regs+bars+Cl).copy().set_index(["iso3","year"])
    f = f"{dep} ~ 1 + " + " + ".join(regs+bars+Cl)
    return RandomEffects.from_formula(f, data=d).fit(cov_type="clustered", cluster_entity=True)

s1 = fe("gdp_pc_growth", ["F_l","G_l","FG_l"]+Cl)
s4 = fe("gdp_pc_growth", ["F_l","G_l","FG_l","MI_l","F_MI_l","G_MI_l","FG_MI_l"]+Cl)
print("BASELINE FE   beta3 (FxG)    = %+.4f%s   N=%d" % (s1.params["FG_l"], star(s1.pvalues["FG_l"]), s1.nobs))
print("BASELINE FE   delta4 (FxGxMI)= %+.4f%s   N=%d" % (s4.params["FG_MI_l"], star(s4.pvalues["FG_MI_l"]), s4.nobs))

BASELINE FE   beta3 (FxG)    = +1.0956**   N=1380
BASELINE FE   delta4 (FxGxMI)= -0.6092   N=1380


In [ ]:
print("MUNDLAK / CORRELATED RANDOM EFFECTS")
print("  within = coef on x_it ; between = within + coef on country-mean ; p = test(mean=0)\n")
out = []
for lbl, regs, keys in [("Spec1", ["F_l","G_l","FG_l"], ["F_l","G_l","FG_l"]),
                        ("Spec4", ["F_l","G_l","FG_l","MI_l","F_MI_l","G_MI_l","FG_MI_l"], ["FG_l","FG_MI_l"])]:
    r = cre("gdp_pc_growth", regs)
    for v in keys:
        wi, lam, pl = r.params[v], r.params[v+"_bar"], r.pvalues[v+"_bar"]
        out.append((lbl, v, round(wi,3), round(wi+lam,3), f"{pl:.3f}{star(pl)}"))
print(pd.DataFrame(out, columns=["spec","var","within","between","p(within≠between)"]).to_string(index=False))

MUNDLAK / CORRELATED RANDOM EFFECTS
  within = coef on x_it ; between = within + coef on country-mean ; p = test(mean=0)

 spec     var  within  between p(within≠between)
Spec1     F_l  0.3170   0.2050             0.822
Spec1     G_l  0.0110   1.3520             0.521
Spec1    FG_l  0.7290  -0.7430          0.003***
Spec4    FG_l  1.1610  -1.0410          0.008***
Spec4 FG_MI_l -0.6910   0.8590             0.136


**How to read this.** If β₃ (F×G) is positive *within* but negative *between* and the
two differ significantly, your positive headline is identified off countries that deepen
both technologies over time — not off a rich-country cross-sectional confound. That makes
the between-estimator sign flip in your robustness table *expected*, not contradictory.
If G's growth effect lives entirely in the *between* column (within ≈ 0), country FE can't
use G at all — which is exactly what Section 5 fixes.

## 3. Use the between dimension on purpose — region×year FE

Replacing country FE with region+year FE retains G's cross-sectional variation while still
absorbing global shocks (COVID). Report this alongside the within results to bracket the range.

In [ ]:
def regfe(dep, regs):
    d = df.dropna(subset=[dep]+regs+Cl+["region"]).copy().set_index(["iso3","year"])
    dums = pd.get_dummies(d["region"], prefix="rg", drop_first=True).astype(float)
    X = pd.concat([d[regs+Cl], dums], axis=1)
    dd = pd.concat([d[[dep]], X], axis=1)
    f = f"{dep} ~ 1 + " + " + ".join(X.columns) + " + TimeEffects"
    return PanelOLS.from_formula(f, data=dd, drop_absorbed=True).fit(cov_type="clustered", cluster_entity=True)

r1 = regfe("gdp_pc_growth", ["F_l","G_l","FG_l"])
r4 = regfe("gdp_pc_growth", ["F_l","G_l","FG_l","MI_l","F_MI_l","G_MI_l","FG_MI_l"])
print("REGION+YEAR FE  beta3 (FxG)    = %+.4f%s   G_l = %+.4f%s (retained)" %
      (r1.params["FG_l"], star(r1.pvalues["FG_l"]), r1.params["G_l"], star(r1.pvalues["G_l"])))
print("REGION+YEAR FE  delta4 (FxGxMI)= %+.4f%s" % (r4.params["FG_MI_l"], star(r4.pvalues["FG_MI_l"])))

REGION+YEAR FE  beta3 (FxG)    = -0.4145*   G_l = +1.0803*** (retained)
REGION+YEAR FE  delta4 (FxGxMI)= +0.5417


## 4. Rebuild F — 4 Findex components + annual IMF FAS  *(needs downloads)*

The single highest-leverage fix. Download both, then edit the CONFIG block to match the
exact column names in your files.

**Download:**
- Global Findex country CSV (waves 2011/14/17/21/24, 4 components):
  https://www.worldbank.org/en/publication/globalfindex/download-data
  (API: Data360 dataset `WB_FINDEX` — https://data360.worldbank.org/en/dataset/WB_FINDEX)
- IMF Financial Access Survey — **annual 2004–2024, 163 economies**:
  https://data.imf.org/en/datasets/IMF.STA:FAS  (or https://db.nomics.world/IMF/FAS)

Findex level (waves) sets the anchor; the within-country z of the annual FAS series adds
the year-to-year shape between waves.

In [ ]:
# ============================================================================
#  Rebuild F : 4 Findex components (level/anchor) + annual IMF FAS (shape)
#  Self-contained. Auto-classifies the two input files by their columns,
#  so the Findex/FAS files can be uploaded in any order without a swap error.
# ============================================================================
import numpy as np, pandas as pd, glob
def z(s): s = pd.to_numeric(s, errors="coerce"); return (s - s.mean()) / s.std()

# ---- config -----------------------------------------------------------------
FINDEX_ISO, FINDEX_YEAR = "codewb", "year"     # codewb = ISO3 ; countrynewwb is the NAME
FINDEX_COMPONENTS = {                           # the 4 components of Eq.25
    "account":  "account_t_d",   # account ownership
    "digpay":   "g20_any",       # made OR received a digital payment
    "saved":    "fin16",         # saved at a financial institution
    "borrowed": "fin22a",        # borrowed from a financial institution
}
FAS_SERIES = ["FA37N", "FA67N", "FA57N"]        # mobile/internet-banking txns; MM accounts; MM agent outlets

# ---- 0. load + auto-classify the two input files ----------------------------
def classify(path):
    cols = set(pd.read_csv(path, nrows=0).columns)
    if "SERIES_CODE" in cols:              return "fas"
    if {"codewb","countrynewwb"} & cols:   return "findex"
    return None

paths = {}
for p in glob.glob("*.csv"):
    r = classify(p)
    if r and r not in paths: paths[r] = p

for role, label in [("findex","Global Findex CSV"), ("fas","IMF FAS dump CSV")]:
    while role not in paths:
        from google.colab import files
        print(f"\n>>> Upload your {label}:")
        up = files.upload()
        for name in up:
            r = classify(name)
            if r: paths[r] = name
            print(f"    '{name}' detected as: {r or 'UNKNOWN — wrong file?'}")

fx      = pd.read_csv(paths["findex"])
fas_raw = pd.read_csv(paths["fas"])
print("Findex:", paths["findex"], fx.shape, "| FAS:", paths["fas"], fas_raw.shape)

# ---- 1. Findex: average the 4 z-scored components on wave years -------------
fx = fx.rename(columns={FINDEX_ISO:"iso3", FINDEX_YEAR:"year"})
zc = []
for name, col in FINDEX_COMPONENTS.items():
    if col in fx.columns: fx[f"z_{name}"] = z(fx[col]); zc.append(f"z_{name}")
    else: print(f"  [warn] Findex column {col!r} not found -> skipping {name}")
assert zc, "No Findex components mapped — check FINDEX_COMPONENTS against your header."
fx["F_findex"] = fx[zc].mean(axis=1)
print(f"  mapped {len(zc)} Findex components:", [c[2:] for c in zc])

# ---- 2. FAS: build the annual within-country shape --------------------------
fas_raw["iso3"]     = fas_raw["SERIES_CODE"].str.split(".").str[0]
fas_raw["ind_code"] = fas_raw["SERIES_CODE"].str.split(".").str[1]
yrs = [str(y) for y in range(2004, 2026)]
sub = fas_raw[fas_raw["ind_code"].isin(FAS_SERIES)]
fas = (sub.melt(id_vars=["iso3","ind_code"], value_vars=yrs, var_name="year", value_name="v")
          .assign(v=lambda d: pd.to_numeric(d["v"], errors="coerce"),
                  year=lambda d: d["year"].astype(int))
          .dropna(subset=["v"])
          .pivot_table(index=["iso3","year"], columns="ind_code", values="v", aggfunc="first")
          .reset_index())
fas.columns.name = None
got = [c for c in FAS_SERIES if c in fas.columns]
fas["F_fas"] = fas[got].apply(z).mean(axis=1)
print(f"  FAS series used:", got, "| countries:", fas.iso3.nunique())

# ---- 3. combine: Findex level (anchor) + FAS within-country shape -----------
P = (df.merge(fx[["iso3","year","F_findex"]], on=["iso3","year"], how="left")
       .merge(fas[["iso3","year","F_fas"]],   on=["iso3","year"], how="left")
       .sort_values(["iso3","year"]).copy())
P["F_fas_w"]  = P.groupby("iso3")["F_fas"].transform(z)
P["F_anchor"] = P.groupby("iso3")["F_findex"].transform(lambda s: s.interpolate().ffill().bfill())
P["F_new"]    = P["F_anchor"] + P["F_fas_w"].fillna(0) * P.groupby("iso3")["F_anchor"].transform("std")
P["F"]  = z(P["F_new"]); P["FG"] = P["F"] * P["G"]
for v in ["F","FG"]: P[v+"_l"] = P.groupby("iso3")[v].shift(1)

w = P.groupby("iso3")["F"].std().mean(); b = P.groupby("iso3")["F"].mean().std()
print(f"\n  NEW F within/between SD ratio = {w/b:.3f}   (was 0.281)")
df = P


>>> Upload your Global Findex CSV:


Saving GlobalFindexDatabase2025 (2).csv to GlobalFindexDatabase2025 (2).csv
    'GlobalFindexDatabase2025 (2).csv' detected as: findex

>>> Upload your IMF FAS dump CSV:


Saving IMF data.csv to IMF data.csv
    'IMF data.csv' detected as: fas
Findex: GlobalFindexDatabase2025 (2).csv (8564, 437) | FAS: IMF data.csv (26896, 31)
  mapped 4 Findex components: ['account', 'digpay', 'saved', 'borrowed']
  FAS series used: ['FA37N', 'FA67N', 'FA57N'] | countries: 141

  NEW F within/between SD ratio = 0.634   (was 0.281)


In [ ]:
# ---- CONFIG: match these to your downloaded files ----
FINDEX_CSV = "findex_country.csv"
FAS_CSV    = "imf_fas.csv"
FINDEX_COMPONENTS = {            # the 4 components of Eq.25 -> your file's column names
    "account":  "account.t.d",
    "digpay":   "fin14a.d",      # made OR received a digital payment
    "saved":    "fin16.d",       # saved at a financial institution
    "borrowed": "fin22a.d",      # borrowed from a financial institution
}
FINDEX_ISO, FINDEX_YEAR = "countrynewwb", "year"
FAS_SERIES = ["FCMOAA_NUM","FCMIBT_NUM"]   # mobile-money agents/100k; mobile+internet banking txns
FAS_ISO, FAS_YEAR = "iso3", "year"
# -------------------------------------------------------

def z(s): s = pd.to_numeric(s, errors="coerce"); return (s - s.mean())/s.std()

def rebuild_F(df):
    fx = pd.read_csv(FINDEX_CSV).rename(columns={FINDEX_ISO:"iso3", FINDEX_YEAR:"year"})
    zc = []
    for name, col in FINDEX_COMPONENTS.items():
        if col in fx.columns: fx[f"z_{name}"] = z(fx[col]); zc.append(f"z_{name}")
        else: print(f"  [warn] missing Findex col {col!r} -> skip {name}")
    fx["F_findex"] = fx[zc].mean(axis=1)
    P = df.merge(fx[["iso3","year","F_findex"]], on=["iso3","year"], how="left").sort_values(["iso3","year"])
    try:
        fa = pd.read_csv(FAS_CSV).rename(columns={FAS_ISO:"iso3", FAS_YEAR:"year"})
        have = [c for c in FAS_SERIES if c in fa.columns]
        fa["F_fas"] = fa[have].apply(z).mean(axis=1)
        P = P.merge(fa[["iso3","year","F_fas"]], on=["iso3","year"], how="left")
        P["F_fas_w"] = P.groupby("iso3")["F_fas"].transform(z)
        P["F_anchor"] = P.groupby("iso3")["F_findex"].transform(lambda s: s.interpolate().ffill().bfill())
        P["F_new"] = P["F_anchor"] + P["F_fas_w"].fillna(0)*P.groupby("iso3")["F_anchor"].transform("std")
    except FileNotFoundError:
        print("  [warn] FAS file not found -> wave-only interpolation")
        P["F_new"] = P.groupby("iso3")["F_findex"].transform(lambda s: s.interpolate())
    P["F"]  = z(P["F_new"]); P["FG"] = P["F"]*P["G"]
    for v in ["F","FG"]: P[v+"_l"] = P.groupby("iso3")[v].shift(1)
    w = P.groupby("iso3")["F"].std().mean(); b = P.groupby("iso3")["F"].mean().std()
    print(f"  NEW F within/between SD ratio = {w/b:.3f}   (was 0.281)")
    return P

# Uncomment once the source files are present:
# df = rebuild_F(df)
print("Edit CONFIG, place the two files in the working dir, then uncomment the last line.")

Edit CONFIG, place the two files in the working dir, then uncomment the last line.


## 5. Rebuild G — annual GARI vintages + AI patents  *(needs downloads)*

Turns the zero-variation readiness pillar into a moving series and adds a second activity pillar.

**Download:**
- GARI annual vintages (2017, 2019, 2020, 2021, 2022, 2023, 2024, 2025; CC BY-SA 4.0):
  https://oxfordinsights.com/ai-readiness/  — stack into `gari_panel.csv` (iso3, year, gari_total)
- AI patents by country, annual: OECD.AI https://oecd.ai/ or Stanford AI Index
  — save as `ai_patents.csv` (iso3, year, ai_patents)

In [ ]:
# ============================================================================
#  Rebuild G : upload GARI vintages + AI patents, then rebuild G
#  Run this AFTER the F-rebuild cell (it uses the rebuilt F to recompute FxG).
# ============================================================================
import numpy as np, pandas as pd, re
from google.colab import files
try: import pycountry
except Exception:
    import subprocess, sys; subprocess.run([sys.executable,"-m","pip","-q","install","pycountry"]); import pycountry
def z(s): s = pd.to_numeric(s, errors="coerce"); return (s - s.mean())/s.std()

# ---- optional manual column overrides (leave None to auto-detect) -----------
GARI_COLS = {"country": None, "year": None, "score": None}
PAT_COLS  = {"country": None, "year": None, "value": None}

def _norm(s): return re.sub(r"[^a-z0-9]+"," ",str(s).lower()).strip()
_ALIAS = {_norm(k):v for k,v in {
    "turkey":"TUR","korea rep":"KOR","south korea":"KOR","korea dem":"PRK","north korea":"PRK",
    "russia":"RUS","iran":"IRN","egypt":"EGY","venezuela":"VEN","vietnam":"VNM","laos":"LAO",
    "syria":"SYR","slovakia":"SVK","brunei":"BRN","swaziland":"SWZ","macedonia":"MKD",
    "kyrgyz republic":"KGZ","congo dem rep":"COD","congo rep":"COG","yemen":"YEM",
    "hong kong":"HKG","gambia":"GMB","cape verde":"CPV"}.items()}
def to_iso3(x):
    s = str(x).strip()
    if not s or s.lower()=="nan": return None
    if _norm(s) in _ALIAS: return _ALIAS[_norm(s)]
    for cand in (s, s.split(",")[0].strip(), s.split("(")[0].strip()):
        try: return pycountry.countries.lookup(cand).alpha_3
        except Exception: pass
    try: return pycountry.countries.search_fuzzy(s.split(",")[0])[0].alpha_3
    except Exception: return None

def _pick(cols, subs, override=None):
    if override: return override
    low = {c.lower(): c for c in cols}
    for s in subs:
        for lc, orig in low.items():
            if s in lc: return orig
    return None

def normalize(fname, kind, ov):
    raw = pd.read_csv(fname)
    print(f"    '{fname}' cols: {list(raw.columns)[:10]}{' ...' if len(raw.columns)>10 else ''}")
    ccol = _pick(raw.columns, ["iso3","iso","country","economy","nation","area"], ov["country"])
    vsub = ["gari_total","total score","total","overall","score","readiness","index"] if kind=="gari" \
           else ["ai_patents","patent","value","count","number"]
    vcol = _pick(raw.columns, vsub, ov["score" if kind=="gari" else "value"])
    ycol = _pick(raw.columns, ["year","edition","date"], ov["year"])
    if not ccol or not vcol:
        raise ValueError(f"{fname}: couldn't find country/value column (country={ccol}, value={vcol}). "
                         f"Set the override dict at top of the cell.")
    print(f"      using country='{ccol}', value='{vcol}', year='{ycol}'")
    out = pd.DataFrame({"iso3": raw[ccol].map(to_iso3),
                        "val":  pd.to_numeric(raw[vcol], errors="coerce")})
    if ycol:
        out["year"] = pd.to_numeric(raw[ycol].astype(str).str.extract(r"(\d{4})")[0], errors="coerce")
    else:
        m = re.search(r"(20\d{2})", fname); out["year"] = int(m.group(1)) if m else np.nan
        print(f"      no year column -> using {out['year'].iloc[0]} from filename")
    out = out.dropna(subset=["iso3","year","val"]); out["year"] = out["year"].astype(int)
    return out

# ---- 1. GARI (select one or several vintage files at once) ------------------
print(">>> Upload your GARI file(s):")
up = files.upload()
gari = pd.concat([normalize(f,"gari",GARI_COLS) for f in up], ignore_index=True)
gari = gari.rename(columns={"val":"gari_total"})[["iso3","year","gari_total"]]
gari = gari.sort_values(["iso3","year"]).drop_duplicates(["iso3","year"], keep="last")
gari.to_csv("gari_panel.csv", index=False)
print(f"  gari_panel.csv: {gari.shape} | countries {gari.iso3.nunique()} | years {sorted(gari.year.unique())}")

# ---- 2. AI patents (optional — press Cancel on the picker to skip) ----------
print("\n>>> Upload your AI-patents file (Cancel to skip, uses publications only):")
try:
    up = files.upload()
    if up:
        pat = pd.concat([normalize(f,"pat",PAT_COLS) for f in up], ignore_index=True)
        pat = pat.rename(columns={"val":"ai_patents"})[["iso3","year","ai_patents"]]
        pat = pat.groupby(["iso3","year"], as_index=False)["ai_patents"].sum()
        pat.to_csv("ai_patents.csv", index=False)
        print(f"  ai_patents.csv: {pat.shape} | countries {pat.iso3.nunique()}")
    else: print("  (skipped)")
except Exception as e: print("  skipped patents:", e)

# ---- 3. rebuild G -----------------------------------------------------------
def rebuild_G(df):
    P = df.sort_values(["iso3","year"]).copy()
    g = pd.read_csv("gari_panel.csv")
    P = P.merge(g, on=["iso3","year"], how="left")
    P["Z_readiness_tv"] = z(P.groupby("iso3")["gari_total"].transform(lambda s: s.ffill().bfill()))
    try:
        pat = pd.read_csv("ai_patents.csv")
        P = P.merge(pat, on=["iso3","year"], how="left")
        P["Z_patents"] = z(np.log1p(P["ai_patents"]))
        P["Z_activity_tv"] = P[["Z_activity","Z_patents"]].mean(axis=1)
    except FileNotFoundError:
        print("  [info] no patents file -> activity = publications only")
        P["Z_activity_tv"] = P["Z_activity"]
    P["G"]  = P[["Z_readiness_tv","Z_activity_tv","Z_adoption"]].mean(axis=1)
    P["FG"] = P["F"] * P["G"]
    for v in ["G","FG"]: P[v+"_l"] = P.groupby("iso3")[v].shift(1)
    w = P.groupby("iso3")["G"].std().mean(); b = P.groupby("iso3")["G"].mean().std()
    print(f"\n  NEW G within/between SD ratio = {w/b:.3f}   (was 0.092)")
    return P

df = rebuild_G(df)

>>> Upload your GARI file(s):


Saving gari_2024.csv to gari_2024.csv
    'gari_2024.csv' cols: ['iso3', 'year', 'gari_total']
      using country='iso3', value='gari_total', year='year'
  gari_panel.csv: (188, 3) | countries 188 | years [np.int64(2024)]

>>> Upload your AI-patents file (Cancel to skip, uses publications only):


  (skipped)
  [info] no patents file -> activity = publications only

  NEW G within/between SD ratio = 0.092   (was 0.092)


## 6. Re-estimate on the rebuilt panel

After running Sections 4 and 5 (so `df` now holds the rebuilt `F`, `G`, `FG` and their
lags), re-run the diagnosis and the Mundlak block to see the payoff: the within/between
ratios should rise, the within–between gap on β₃ should shrink, and δ₄ gets its first
properly-powered test.

In [ ]:
# 1) variation check
for v in ["F","G"]:
    w,b,r = wb_ratio(v); print(f"{v}: within/between = {r:.3f}")

# 2) recompute Mundlak means on the rebuilt lags, then re-run
for v in KEY:
    if v in df.columns: df[v+"_bar"] = df.groupby("iso3")[v].transform("mean")

print()
print("BASELINE FE (rebuilt)   beta3 = %+.4f%s" % (lambda r:(r.params['FG_l'], star(r.pvalues['FG_l'])))(fe("gdp_pc_growth", ["F_l","G_l","FG_l"]+Cl)))
out = []
for lbl, regs, keys in [("Spec1", ["F_l","G_l","FG_l"], ["F_l","G_l","FG_l"]),
                        ("Spec4", ["F_l","G_l","FG_l","MI_l","F_MI_l","G_MI_l","FG_MI_l"], ["FG_l","FG_MI_l"])]:
    r = cre("gdp_pc_growth", regs)
    for v in keys:
        wi, lam, pl = r.params[v], r.params[v+"_bar"], r.pvalues[v+"_bar"]
        out.append((lbl, v, round(wi,3), round(wi+lam,3), f"{pl:.3f}{star(pl)}"))
print(pd.DataFrame(out, columns=["spec","var","within","between","p(diff)"]).to_string(index=False))

F: within/between = 0.634
G: within/between = 0.092

BASELINE FE (rebuilt)   beta3 = +0.3602**
 spec     var  within  between  p(diff)
Spec1     F_l  0.3840   2.2830  0.032**
Spec1     G_l  6.0550   3.2380    0.278
Spec1    FG_l  0.6600  -1.6730 0.000***
Spec4    FG_l  0.8040  -2.7160 0.000***
Spec4 FG_MI_l -0.0180   1.5720    0.170


In [ ]:
for v in keys:
    wi, pw  = r.params[v], r.pvalues[v]          # within coef + its own p-value
    lam, pl = r.params[v+"_bar"], r.pvalues[v+"_bar"]
    out.append((lbl, v, round(wi,3), star(pw), round(wi+lam,3), f"{pl:.3f}{star(pl)}"))
print(pd.DataFrame(out, columns=["spec","var","within","within_p","between","p(diff)"]).to_string(index=False))

 spec     var  within within_p  between  p(diff)
Spec1     F_l  0.3840   2.2830  0.032**     None
Spec1     G_l  6.0550   3.2380    0.278     None
Spec1    FG_l  0.6600  -1.6730 0.000***     None
Spec4    FG_l  0.8040  -2.7160 0.000***     None
Spec4 FG_MI_l -0.0180   1.5720    0.170     None
Spec4    FG_l  0.8040      ***  -2.7160 0.000***
Spec4 FG_MI_l -0.0180            1.5720    0.170


In [ ]:
# ===== Export the rebuilt panel =====
# sanity check: confirm df actually holds the rebuilt F (not the original)
assert "F_findex" in df.columns, "This df doesn't look rebuilt — run the F-rebuild cell first."
w = df.groupby("iso3")["F"].std().mean() / df.groupby("iso3")["F"].mean().std()
print(f"F within/between ratio in this df = {w:.3f}  (rebuilt should be ~0.63, original ~0.28)")

# optional: drop the scratch columns from the rebuild, keep the clean panel
scratch = ["F_findex","F_fas","F_fas_w","F_anchor","F_new"]
out = df.drop(columns=[c for c in scratch if c in df.columns])

out.to_csv("final_panel_F_rebuilt.csv", index=False)
print("wrote final_panel_F_rebuilt.csv:", out.shape)

from google.colab import files
files.download("final_panel_F_rebuilt.csv")

F within/between ratio in this df = 0.634  (rebuilt should be ~0.63, original ~0.28)
wrote final_panel_F_rebuilt.csv: (10424, 147)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
### Priority order (return on effort)
1. **Rebuild F** (Section 4) — fixes the attenuation behind nearly every fragile coefficient.
2. **Make G time-varying** (Section 5) — without it, country FE can't use G at all.
3. **Mundlak + region×year FE** (Sections 2–3) — free, runs today; reframes the between-flip.
4. Longer panel + discrete-time survival model for the MIT-escape / vulnerability tests.
5. De-construct the displacement term; fix or retire the weak IV.